# Geometric AR order estimation via minimum width

This notebook simulates a stationary AR($p$) process
$$X_t=\phi_1X_{t-1}+\cdots+\phi_pX_{t-p}+\varepsilon_t,$$
forms forward sliding-window clouds $Y_t^k=(X_t,\ldots,X_{t+k-1})\in\mathbb R^k$, and estimates
$$\widehat m_k=\min_{u\in S^{k-1}}\widehat w_k(u),\qquad \widehat w_k(u)=\max_t\langle u,Y_t^k\rangle-\min_t\langle u,Y_t^k\rangle.$$

At $k=p+1$, the normalized AR-recurrence direction is proportional to
$$u_{\rm AR}=(-\phi_p,-\phi_{p-1},\ldots,-\phi_1,1),$$
and $\langle u_{\rm AR},Y_t^{p+1}\rangle=\varepsilon_{t+p}/\|u_{\rm AR}\|$. If innovations are small relative to process variation, this narrow direction first appears at $k=p+1$ and may produce a kink. This is a finite-sample diagnostic, not yet a calibrated order estimator.

## Numerical minimization actually used

I parameterize $S^{k-1}$ by $k-1$ hyperspherical angles. The notebook first runs SciPy's **differential evolution** (a bounded population-based global search), then polishes its best point using bounded **Powell** minimization. The objective is nonconvex and nonsmooth where the extremal observation changes, so a derivative-free global-to-local strategy is appropriate. For $k=1$, $S^0=\{-1,+1\}$ and the answer is exactly the sample range.

This is numerical, not an exact convex-hull minimum-width algorithm. For larger $k$, increase `de_maxiter` and `de_popsize` or repeat with several optimizer seeds.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize
from scipy.stats import t as student_t_dist

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=4, suppress=True)

In [ ]:
def companion_spectral_radius(phi):
    phi = np.asarray(phi, dtype=float)
    A = np.zeros((len(phi), len(phi)))
    A[0] = phi
    if len(phi) > 1:
        A[1:, :-1] = np.eye(len(phi) - 1)
    return np.max(np.abs(np.linalg.eigvals(A)))


def automatic_stationary_ar_coefficients(order, first_pacf=0.95, decay=-0.55):
    """Generate stationary AR coefficients from decaying partial autocorrelations.

    Uses kappa_j = first_pacf * decay**(j-1). Since every |kappa_j| < 1,
    the Levinson recursion guarantees stationarity.
    """
    if not isinstance(order, (int, np.integer)) or order < 1:
        raise ValueError("order must be a positive integer")
    if not (abs(first_pacf) < 1 and abs(decay) < 1):
        raise ValueError("first_pacf and decay must have absolute value below one")
    pacf = first_pacf * decay**np.arange(order)
    phi = np.array([pacf[0]])
    for m in range(2, order + 1):
        old_phi = phi.copy()
        reflection = pacf[m - 1]
        phi = np.empty(m)
        phi[:-1] = old_phi - reflection * old_phi[::-1]
        phi[-1] = reflection
    return phi


def simulate_ar(phi, n=2_000, noise_scale=0.25, innovation="uniform",
                student_df=2.0, burn_in=1_000, seed=2026):
    """Simulate a zero-mean stationary AR(p).

    noise_scale is the innovation SD for uniform/Gaussian noise. Student-t noise
    is scaled to the same MAD as N(0, noise_scale**2), even when variance is infinite.
    """
    phi = np.asarray(phi, dtype=float)
    p = len(phi)
    radius = companion_spectral_radius(phi)
    if radius >= 1:
        raise ValueError(f"AR coefficients are not stationary (spectral radius={radius:.3f}).")
    rng = np.random.default_rng(seed)
    size = n + burn_in
    if innovation == "uniform":
        eps = rng.uniform(-np.sqrt(3)*noise_scale, np.sqrt(3)*noise_scale, size)
    elif innovation == "gaussian":
        eps = rng.normal(0.0, noise_scale, size)
    elif innovation == "student_t":
        if student_df <= 0:
            raise ValueError("student_df must be positive")
        target_mad = 0.6744897501960817 * noise_scale
        t_scale = target_mad / student_t_dist.ppf(0.75, student_df)
        eps = rng.standard_t(student_df, size=size) * t_scale
    else:
        raise ValueError("innovation must be 'uniform', 'gaussian', or 'student_t'")
    x = np.zeros(size)
    for t in range(p, size):
        x[t] = phi @ x[t - np.arange(1, p + 1)] + eps[t]
    return x[burn_in:]


def sliding_window_embedding(x, k):
    """Rows are (x_t,...,x_{t+k-1}), with shape (n-k+1,k)."""
    return np.lib.stride_tricks.sliding_window_view(np.asarray(x), k).copy()


def angles_to_unit_vector(theta, k):
    """Standard hyperspherical coordinates in R^k."""
    if k == 1:
        return np.ones(1)
    theta = np.asarray(theta)
    u = np.empty(k)
    sine_product = 1.0
    for j in range(k - 1):
        u[j] = sine_product * np.cos(theta[j])
        sine_product *= np.sin(theta[j])
    u[-1] = sine_product
    return u


def directional_width(Y, u):
    return np.ptp(Y @ u)


def directional_quantile_width(Y, u, alpha=0.05):
    """Empirical central quantile width using two order statistics."""
    if not 0.0 <= alpha < 0.5:
        raise ValueError("alpha must satisfy 0 <= alpha < 0.5")
    projection = Y @ u
    n_projection = len(projection)
    lower_index = int(np.floor(alpha * (n_projection - 1)))
    upper_index = int(np.ceil((1.0 - alpha) * (n_projection - 1)))
    partitioned = np.partition(projection, (lower_index, upper_index))
    return float(partitioned[upper_index] - partitioned[lower_index])


def estimate_minimum_quantile_width(Y, alpha=0.05, seed=0,
                                    de_maxiter=50, de_popsize=6, polish=True):
    """Minimize the empirical alpha-to-(1-alpha) quantile width."""
    k = Y.shape[1]
    if k == 1:
        return dict(m_hat=directional_quantile_width(Y, np.ones(1), alpha),
                    u_hat=np.ones(1), global_success=True, local_success=True)
    bounds = [(0.0, np.pi)]*(k - 2) + [(0.0, 2.0*np.pi)]
    objective = lambda theta: directional_quantile_width(
        Y, angles_to_unit_vector(theta, k), alpha)
    global_result = differential_evolution(
        objective, bounds=bounds, seed=seed, maxiter=de_maxiter,
        popsize=de_popsize, tol=1e-6, updating="immediate", workers=1, polish=False)
    best_x, best_fun = global_result.x, global_result.fun
    local_success = True
    if polish:
        local_result = minimize(objective, best_x, method="Powell", bounds=bounds,
                                options={"xtol": 1e-8, "ftol": 1e-8, "maxiter": 800})
        local_success = bool(local_result.success)
        if local_result.fun < best_fun:
            best_x, best_fun = local_result.x, local_result.fun
    return dict(m_hat=float(best_fun), u_hat=angles_to_unit_vector(best_x, k),
                global_success=bool(global_result.success), local_success=local_success)


def estimate_minimum_width(Y, seed=0, de_maxiter=70, de_popsize=8, polish=True):
    """Differential evolution in angles, optionally followed by bounded Powell."""
    k = Y.shape[1]
    if k == 1:
        return dict(m_hat=float(np.ptp(Y[:, 0])), u_hat=np.ones(1),
                    global_success=True, local_success=True)
    bounds = [(0.0, np.pi)]*(k - 2) + [(0.0, 2.0*np.pi)]
    objective = lambda theta: directional_width(Y, angles_to_unit_vector(theta, k))
    global_result = differential_evolution(
        objective, bounds=bounds, seed=seed, maxiter=de_maxiter,
        popsize=de_popsize, tol=1e-6, updating="immediate", workers=1, polish=False)
    best_x, best_fun = global_result.x, global_result.fun
    local_success = True
    if polish:
        local_result = minimize(objective, best_x, method="Powell", bounds=bounds,
                                options={"xtol": 1e-8, "ftol": 1e-8, "maxiter": 800})
        local_success = bool(local_result.success)
        if local_result.fun < best_fun:
            best_x, best_fun = local_result.x, local_result.fun
    return dict(m_hat=float(best_fun), u_hat=angles_to_unit_vector(best_x, k),
                global_success=bool(global_result.success), local_success=local_success)


def select_ar_order_bic(x, q_max=7):
    """Select AR order q in 0,...,q_max by conditional-least-squares BIC.

    Every candidate uses the common response segment x[q_max:], and an intercept
    is included. BIC(q) = n_eff*log(RSS_q/n_eff) + (q+1)*log(n_eff).
    """
    x = np.asarray(x, dtype=float)
    if not isinstance(q_max, (int, np.integer)) or q_max < 0 or q_max >= len(x):
        raise ValueError("q_max must be an integer satisfying 0 <= q_max < len(x)")
    y = x[q_max:]
    n_eff = len(y)
    bic_values = []
    for q in range(q_max + 1):
        columns = [np.ones(n_eff)]
        columns.extend(x[q_max-lag:len(x)-lag] for lag in range(1, q + 1))
        design = np.column_stack(columns)
        beta = np.linalg.lstsq(design, y, rcond=None)[0]
        residuals = y - design @ beta
        sigma2 = max(float(residuals @ residuals / n_eff), np.finfo(float).tiny)
        bic_values.append(n_eff*np.log(sigma2) + design.shape[1]*np.log(n_eff))
    return int(np.argmin(bic_values)), np.asarray(bic_values)


def ar_recurrence_direction(phi):
    v = np.r_[-np.asarray(phi)[::-1], 1.0]
    return v / np.linalg.norm(v)

## Experiment settings

Choose `p` and `k_max` directly below. Coefficients are generated automatically using the deterministic partial-autocorrelation rule $\kappa_j=0.95(-0.55)^{j-1}$ and converted to AR coefficients by the Levinson recursion. Since $|\kappa_j|<1$, this produces a stationary AR($p$) model with a nonzero final coefficient. Uniform innovations are the default because a statistic based on extrema is more stable with bounded noise; switch to `"gaussian"` to assess tail sensitivity.

In [ ]:
# Main structural choices
p = 3
k_max = 7

# Coefficients are generated automatically from p.
phi = automatic_stationary_ar_coefficients(p)

n = 1_200
noise_scale = 0.25
innovation = "uniform"       # or "gaussian"
simulation_seed = 2026
optimizer_seed = 17
de_maxiter = 70
de_popsize = 8

if k_max < 1 or k_max > n:
    raise ValueError("k_max must satisfy 1 <= k_max <= n")
if k_max < p + 1:
    print(f"Warning: k_max={k_max} does not include the target dimension p+1={p+1}.")

x = simulate_ar(phi, n=n, noise_scale=noise_scale, innovation=innovation, seed=simulation_seed)
embeddings = {k: sliding_window_embedding(x, k) for k in range(1, k_max + 1)}

print(f"AR order p = {p}; expected first recurrence dimension p+1 = {p+1}")
print(f"Automatically generated coefficients phi = {phi}")
print(f"Companion spectral radius = {companion_spectral_radius(phi):.4f}")
print(f"Innovation = {innovation}, innovation SD = {noise_scale}")
print("Embedding shapes:", {k: Y.shape for k, Y in embeddings.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(x, lw=0.8, color="tab:blue")
ax.set(title=f"Simulated AR({p}) process", xlabel="t", ylabel="$X_t$")
plt.show()

## Estimate and plot $\widehat m_k$

The table includes a scale-normalized width. At $k=p+1$, it also compares the optimized direction with the known recurrence direction (up to sign), helping distinguish a genuine AR direction from an optimization artifact.

In [ ]:
results = []
for k, Y in embeddings.items():
    fit = estimate_minimum_width(Y, seed=optimizer_seed+k,
                                 de_maxiter=de_maxiter, de_popsize=de_popsize)
    row = {"k": k, "m_hat": fit["m_hat"],
           "relative_width": fit["m_hat"]/np.ptp(x), "u_hat": fit["u_hat"],
           "DE_converged": fit["global_success"], "Powell_converged": fit["local_success"],
           "alignment_with_AR_direction": np.nan, "width_in_AR_direction": np.nan}
    if k == p + 1:
        u_ar = ar_recurrence_direction(phi)
        row["alignment_with_AR_direction"] = abs(fit["u_hat"] @ u_ar)
        row["width_in_AR_direction"] = directional_width(Y, u_ar)
    results.append(row)

result_table = pd.DataFrame(results)
display(result_table.drop(columns="u_hat").round(5))
print("\nMinimization: differential evolution in hyperspherical angles,")
print("followed by bounded Powell polishing (except the exact k=1 case).")
print("A false DE_converged flag only means the iteration budget was reached;")
print("the best DE point is still retained and polished.")

In [ ]:
ks = result_table["k"].to_numpy()
m_hat = result_table["m_hat"].to_numpy()
relative = result_table["relative_width"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for ax, values, ylabel in [
    (axes[0], m_hat, r"$\widehat m_k$"),
    (axes[1], relative, r"$\widehat m_k / \operatorname{range}(X)$")]:
    ax.plot(ks, values, "o-", lw=2, color="tab:blue", label="estimated minimum width")
    ax.axvline(p+1, color="tab:red", ls="--", lw=1.8, label=r"expected $p+1$")
    ax.set(xticks=ks, xlabel="embedding dimension $k$", ylabel=ylabel)
    ax.legend()
axes[0].set_title(f"Raw width for AR({p})")
axes[1].set_title("Scale-normalized width")
plt.show()

if p + 1 <= k_max:
    before, at_kink = m_hat[p-1], m_hat[p]
    print(f"Relative drop from k=p to k=p+1: {(before-at_kink)/before:.1%}")
    print(f"|<u_hat, u_AR>| at k=p+1: {result_table.loc[p, 'alignment_with_AR_direction']:.5f}")

## Interpretation and limitations

Evidence for the proposed mechanism consists of both (i) a marked downward change at $k=p+1$ and (ii) high alignment $|\langle\widehat u,u_{\rm AR}\rangle|$ near one. Since $\widehat w_k$ uses sample extrema, its magnitude depends on sample size and innovation tails. A serious order estimator needs uncertainty quantification and a calibrated kink rule; natural next experiments are Monte Carlo repetitions, bootstrap bands, several noise-to-signal ratios, and comparisons with AIC/BIC. Gaussian innovations typically give more variable widths because their observed range grows with sample size.

## Maximum second-difference kink rule

To put the decreasing width curve into the increasing-sequence convention, start with $-\log \widehat m_k$ and apply a cumulative maximum to remove small finite-sample violations of monotonicity. For an increasing curve that changes from steep to flat, define

$$C_k=(a_k-a_{k-1})-(a_{k+1}-a_k)=2a_k-a_{k-1}-a_{k+1},$$

and select $k^*=\operatorname*{arg\,max}_{2\leq k\leq k_{\max}-1}C_k$. The logarithm makes the rule invariant to multiplying all widths by a common scale.

In [ ]:
a_raw = -np.log(m_hat)
a = np.maximum.accumulate(a_raw)  # enforce the assumed nondecreasing sequence
first_differences = np.diff(a)
C_k = first_differences[:-1] - first_differences[1:]
interior_ks = ks[1:-1]
k_star = int(interior_ks[np.argmax(C_k)])

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(interior_ks, C_k, "o-", lw=2, color="tab:purple", label=r"$C_k$")
ax.axvline(k_star, color="black", ls="-", lw=1.8,
           label=fr"selected $k^*={k_star}$")
ax.axvline(p + 1, color="tab:red", ls="--", lw=1.8,
           label=fr"reference $p+1={p+1}$")
ax.set(xticks=interior_ks, xlabel="embedding dimension $k$",
       ylabel=r"$C_k=2a_k-a_{k-1}-a_{k+1}$",
       title=r"Maximum second-difference kink rule for $a_k=-\log \widehat m_k$")
ax.axhline(0.0, color="0.5", lw=0.8)
ax.legend()
plt.show()

print(f"Maximum-second-difference choice: k* = {k_star}")
print(f"Corresponding geometric order estimate: p_hat = k*-1 = {k_star-1}")

## Comparison with alternative automatic rules

All rules below use only the monotone version of $a_k=-\log \widehat m_k$; none uses the AR coefficients. They capture different meanings of a kink:

- **Relative slope drop:** largest proportional reduction after a material incoming gain (at least 10% of the largest gain), preventing tiny plateau fluctuations from dominating.
- **CUSUM:** largest global change in the mean first difference.
- **Kneedle:** largest normalized distance above the diagonal.
- **L-method:** split giving the smallest weighted error of two fitted lines.
- **Plateau onset:** first $k$ after which every remaining gain is at most a chosen fraction of the largest gain.

The plateau fraction is exposed as `plateau_fraction`; `0.10` means that all subsequent log-gains must be no more than 10% of the largest observed absolute log-gain.

In [ ]:
def relative_slope_drop(a, ks, min_gain_fraction=0.10, eps=1e-12):
    gains = np.diff(a)
    candidates = np.asarray(ks)[1:-1]
    scores = (gains[:-1] - gains[1:]) / (np.abs(gains[:-1]) + eps)
    material = gains[:-1] >= min_gain_fraction * np.max(np.abs(gains))
    scores = np.where(material, scores, np.nan)
    return int(candidates[np.nanargmax(scores)]), candidates, scores


def cusum_first_differences(a, ks):
    gains = np.diff(a)
    n_gains = len(gains)
    split_sizes = np.arange(1, n_gains)
    scores = np.array([
        np.sqrt(j * (n_gains - j) / n_gains)
        * (np.mean(gains[:j]) - np.mean(gains[j:]))
        for j in split_sizes
    ])
    candidates = np.asarray(ks)[split_sizes]
    return int(candidates[np.argmax(scores)]), candidates, scores


def kneedle_rule(a, ks, eps=1e-12):
    ks = np.asarray(ks, dtype=float)
    x_normalized = (ks - ks[0]) / (ks[-1] - ks[0] + eps)
    y_normalized = (a - np.min(a)) / (np.max(a) - np.min(a) + eps)
    scores = y_normalized - x_normalized
    interior = np.arange(1, len(ks) - 1)
    selected = interior[np.argmax(scores[interior])]
    return int(ks[selected]), ks, scores


def l_method(a, ks):
    ks = np.asarray(ks, dtype=float)
    candidates, losses = [], []
    for i in range(1, len(ks) - 1):
        x_left, y_left = ks[:i + 1], a[:i + 1]
        x_right, y_right = ks[i:], a[i:]
        fit_left = np.polyval(np.polyfit(x_left, y_left, 1), x_left)
        fit_right = np.polyval(np.polyfit(x_right, y_right, 1), x_right)
        rmse_left = np.sqrt(np.mean((y_left - fit_left)**2))
        rmse_right = np.sqrt(np.mean((y_right - fit_right)**2))
        weighted_rmse = (len(y_left)*rmse_left + len(y_right)*rmse_right) / (len(y_left) + len(y_right))
        candidates.append(ks[i])
        losses.append(weighted_rmse)
    candidates, losses = np.asarray(candidates), np.asarray(losses)
    return int(candidates[np.argmin(losses)]), candidates, losses


def plateau_onset(a, ks, fraction=0.10):
    gains = np.diff(a)
    threshold = fraction * np.max(np.abs(gains))
    for i in range(1, len(ks) - 1):
        if np.all(np.abs(gains[i:]) <= threshold):
            return int(ks[i]), threshold
    return int(ks[-2]), threshold


In [ ]:
plateau_fraction = 0.10

k_second = int(interior_ks[np.argmax(C_k)])
k_relative, relative_ks, relative_scores = relative_slope_drop(a, ks)
k_cusum, cusum_ks, cusum_scores = cusum_first_differences(a, ks)
k_kneedle, kneedle_ks, kneedle_scores = kneedle_rule(a, ks)
k_l_method, l_method_ks, l_method_losses = l_method(a, ks)
k_plateau, plateau_threshold = plateau_onset(a, ks, fraction=plateau_fraction)

rule_choices = {
    "Maximum second difference": k_second,
    "Relative slope drop (material)": k_relative,
    "CUSUM of first differences": k_cusum,
    "Kneedle": k_kneedle,
    "L-method": k_l_method,
    f"Plateau onset ({plateau_fraction:.0%})": k_plateau,
}

rule_table = pd.DataFrame({
    "rule": list(rule_choices),
    "k_star": list(rule_choices.values()),
})
rule_table["p_hat = k_star - 1"] = rule_table["k_star"] - 1
rule_table["equals p+1"] = rule_table["k_star"] == p + 1
display(rule_table)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ks, m_hat, "o-", lw=2.5, color="black", label=r"$\widehat m_k$")
colors = plt.cm.tab10(np.linspace(0, 1, len(rule_choices)))
for (name, selected_k), color in zip(rule_choices.items(), colors):
    ax.axvline(selected_k, color=color, lw=1.4, alpha=0.8, label=f"{name}: {selected_k}")
ax.axvline(p + 1, color="tab:red", ls="--", lw=2.5, label=f"reference p+1: {p+1}")
ax.set(xticks=ks, yscale="log", xlabel="embedding dimension $k$",
       ylabel=r"$\widehat m_k$ (log scale)", title="Comparison of automatic kink rules")
ax.legend(fontsize=8, ncol=2)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5), constrained_layout=True)
diagnostics = [
    (interior_ks, C_k, "Maximum second difference", "maximize", k_second),
    (relative_ks, relative_scores, "Relative slope drop", "maximize", k_relative),
    (cusum_ks, cusum_scores, "CUSUM", "maximize", k_cusum),
    (kneedle_ks, kneedle_scores, "Kneedle distance", "maximize", k_kneedle),
    (l_method_ks, l_method_losses, "L-method weighted RMSE", "minimize", k_l_method),
    (ks[1:], np.abs(first_differences), "Absolute subsequent log-gains", "plateau", k_plateau),
]
for ax, (x_values, scores, title, direction, selected_k) in zip(axes.flat, diagnostics):
    ax.plot(x_values, scores, "o-", lw=1.8)
    ax.axvline(selected_k, color="black", lw=1.5, label=f"k*={selected_k}")
    ax.axvline(p + 1, color="tab:red", ls="--", lw=1.5, label=f"p+1={p+1}")
    if direction == "plateau":
        ax.axhline(plateau_threshold, color="tab:green", ls=":", label="threshold")
    ax.set(title=title, xlabel="k")
    ax.legend(fontsize=8)
plt.show()

## Monte Carlo experiment: 100 independent AR(3) paths

This experiment keeps the automatically generated AR(3) coefficients fixed and simulates 100 independent uniform-innovation paths. It uses $n=800$, $k_{\max}=7$, and a reduced width-optimizer budget of 50 differential-evolution iterations with population size 6. All rule thresholds are frozen before the experiment. The target is $k^*=p+1=4$. A complete run takes roughly 40--50 seconds on the machine used to create this notebook.

In the verified reference run (simulation seeds 20000--20099), the numbers correct out of 100 were: maximum second difference **0**, material relative slope drop **90**, CUSUM **0**, Kneedle **1**, L-method **10**, 10% plateau onset **92**, and classical BIC **100**. BIC estimates AR coefficients internally and is included only as an external benchmark.

In [ ]:
mc_replications = 100
mc_p = 3
mc_k_max = 7
mc_n = 800
mc_de_maxiter = 50
mc_de_popsize = 6
mc_phi = automatic_stationary_ar_coefficients(mc_p)
mc_true_k = mc_p + 1


def apply_all_kink_rules(widths):
    local_ks = np.arange(1, len(widths) + 1)
    local_a = np.maximum.accumulate(-np.log(widths))
    gains = np.diff(local_a)
    curvature = gains[:-1] - gains[1:]
    interior = local_ks[1:-1]
    return {
        "Maximum second difference": int(interior[np.argmax(curvature)]),
        "Relative slope drop (material)": relative_slope_drop(local_a, local_ks)[0],
        "CUSUM of first differences": cusum_first_differences(local_a, local_ks)[0],
        "Kneedle": kneedle_rule(local_a, local_ks)[0],
        "L-method": l_method(local_a, local_ks)[0],
        "Plateau onset (10%)": plateau_onset(local_a, local_ks, fraction=0.10)[0],
    }


mc_rows = []
for replication in range(mc_replications):
    x_mc = simulate_ar(
        mc_phi, n=mc_n, noise_scale=0.25, innovation="uniform",
        seed=20_000 + replication,
    )
    widths_mc = []
    for k in range(1, mc_k_max + 1):
        fit_mc = estimate_minimum_width(
            sliding_window_embedding(x_mc, k),
            seed=30_000 + 100*replication + k,
            de_maxiter=mc_de_maxiter, de_popsize=mc_de_popsize,
        )
        widths_mc.append(fit_mc["m_hat"])
    choices_mc = apply_all_kink_rules(np.asarray(widths_mc))
    bic_order_mc, _ = select_ar_order_bic(x_mc, q_max=mc_k_max)
    choices_mc["BIC (reported as k*=q_hat+1)"] = bic_order_mc + 1
    choices_mc["replication"] = replication
    mc_rows.append(choices_mc)
    if (replication + 1) % 10 == 0:
        print(f"Completed {replication + 1}/{mc_replications}")

mc_selections = pd.DataFrame(mc_rows).set_index("replication")
mc_correct_counts = (mc_selections == mc_true_k).sum().sort_values(ascending=False)
mc_summary = pd.DataFrame({
    f"correct_out_of_{mc_replications}": mc_correct_counts,
    "accuracy": mc_correct_counts / mc_replications,
})
display(mc_summary)

candidate_ks = np.arange(2, mc_k_max)
mc_distribution = pd.DataFrame(
    {name: mc_selections[name].value_counts().reindex(candidate_ks, fill_value=0)
     for name in mc_selections.columns},
    index=candidate_ks,
).T
mc_distribution.columns.name = "selected k_star"
display(mc_distribution)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), constrained_layout=True)
mc_correct_counts.sort_values().plot.barh(ax=axes[0], color="tab:blue")
axes[0].axvline(mc_replications, color="tab:red", ls="--", lw=1.5)
axes[0].set(xlabel=f"correct selections out of {mc_replications}",
            title=fr"Recovery of $k^*=p+1={mc_true_k}$", xlim=(0, mc_replications + 2))

image = axes[1].imshow(mc_distribution.to_numpy(), aspect="auto", cmap="Blues")
axes[1].set(xticks=np.arange(len(candidate_ks)), xticklabels=candidate_ks,
            yticks=np.arange(len(mc_distribution.index)), yticklabels=mc_distribution.index,
            xlabel="selected $k^*$", title="Full selection-frequency distribution")
for row in range(mc_distribution.shape[0]):
    for col in range(mc_distribution.shape[1]):
        value = int(mc_distribution.iloc[row, col])
        axes[1].text(col, row, value, ha="center", va="center",
                     color="white" if value > 50 else "black")
fig.colorbar(image, ax=axes[1], label="frequency")
plt.show()

## Multi-order Monte Carlo experiment

The same frozen protocol was repeated for $p\in\{1,2,3,4,5,7\}$, with 100 independent paths per order and $k_{\max}=7$. The entries below are the number of correct selections out of 100: $k^*=p+1$ for geometric rules and $\widehat q=p$ for BIC. For $p=7$, geometric methods cannot access their target $k^*=8$, while BIC can still consider the true AR order because its candidate set is $q=0,\ldots,7$.

The rapid deterioration for larger $p$ is also connected to the coefficient generator: its partial autocorrelations decay geometrically, so the highest-order contribution becomes progressively weaker and the process has a pronounced lower-order effective structure.

In [ ]:
multi_order_correct_counts = pd.DataFrame(
    {
        1: [100, 100, 100, 100, 100, 100, 100],
        2: [46, 100, 93, 100, 100, 100, 100],
        3: [0, 90, 0, 1, 10, 92, 100],
        4: [0, 24, 0, 0, 0, 30, 99],
        5: [0, 0, 0, 0, 0, 1, 40],
        7: [0, 0, 0, 0, 0, 0, 2],
    },
    index=[
        "Maximum second difference",
        "Relative slope drop (material)",
        "CUSUM of first differences",
        "Kneedle",
        "L-method",
        "Plateau onset (10%)",
        "Classical BIC",
    ],
)
multi_order_correct_counts.columns.name = "AR order p"
display(multi_order_correct_counts)

fig, ax = plt.subplots(figsize=(9, 5.5))
image = ax.imshow(multi_order_correct_counts.to_numpy(), aspect="auto",
                  vmin=0, vmax=100, cmap="YlGn")
ax.set(
    xticks=np.arange(len(multi_order_correct_counts.columns)),
    xticklabels=[1, 2, 3, 4, 5, "7 (target 8 unavailable)"],
    yticks=np.arange(len(multi_order_correct_counts.index)),
    yticklabels=multi_order_correct_counts.index,
    xlabel="AR order $p$",
    title="Correct order-equivalent selections out of 100 simulations",
)
for row in range(multi_order_correct_counts.shape[0]):
    for col in range(multi_order_correct_counts.shape[1]):
        value = int(multi_order_correct_counts.iloc[row, col])
        ax.text(col, row, value, ha="center", va="center",
                color="white" if value >= 70 else "black", fontweight="bold")
fig.colorbar(image, ax=ax, label="correct out of 100")
plt.show()

## Quantile-width experiment for different trimming levels

Replace the full range by the central empirical quantile width

$$\widehat w_{k,\alpha}(u)=\widehat Q_{1-\alpha}(\langle u,Y_t^k\rangle)-\widehat Q_{\alpha}(\langle u,Y_t^k\rangle),$$

and minimize it over the sphere exactly as before. The implementation above uses the corresponding empirical order statistics; $\alpha=0$ therefore reproduces the original range width.

The verified experiment used 100 AR(3) paths, $n=800$, $k_{\max}=7$, the same coefficients and seeds as before, and $\alpha\in\{0,0.01,0.025,0.05,0.10\}$. The table reports correct selections of $k^*=4$ out of 100. BIC remains 100/100 and is independent of $\alpha$. On these clean bounded-innovation data, trimming does not improve recovery; its potential advantage should instead be tested under contamination or heavy tails.

In [ ]:
quantile_alpha_results = pd.DataFrame(
    {
        0.000: [0, 90, 0, 1, 10, 92],
        0.010: [0, 83, 0, 0, 0, 91],
        0.025: [0, 50, 0, 0, 0, 89],
        0.050: [0, 46, 0, 0, 0, 75],
        0.100: [0, 37, 0, 0, 0, 59],
    },
    index=[
        "Maximum second difference",
        "Relative slope drop (material)",
        "CUSUM of first differences",
        "Kneedle",
        "L-method",
        "Plateau onset (10%)",
    ],
)
quantile_alpha_results.columns.name = r"quantile level $\alpha$"
display(quantile_alpha_results)

fig, ax = plt.subplots(figsize=(9, 5.2))
for method in quantile_alpha_results.index:
    ax.plot(quantile_alpha_results.columns, quantile_alpha_results.loc[method],
            "o-", lw=1.8, label=method)
ax.axhline(100, color="black", ls="--", lw=1.5, label="BIC: 100/100")
ax.set(xlabel=r"quantile trimming level $\alpha$",
       ylabel="correct selections out of 100", ylim=(-3, 103),
       title=r"AR(3) order recovery using minimum quantile widths")
ax.legend(fontsize=8, ncol=2)
plt.show()

## Quantile-width experiment with Gaussian innovations

The preceding 100-path AR(3) alpha experiment was repeated with Gaussian innovations and otherwise identical settings, including the simulation seeds. Unlike bounded uniform innovations, Gaussian extrema grow with sample size and fluctuate strongly, so quantile trimming has a clear benefit for the plateau rule. The best tested value was $\alpha=0.10$, with 78 correct selections out of 100, compared with 52 for the untrimmed range. Classical BIC selected the correct order in 99 of the 100 Gaussian paths.

In [ ]:
gaussian_quantile_alpha_results = pd.DataFrame(
    {
        0.000: [0, 54, 0, 17, 15, 52],
        0.010: [0, 43, 0, 3, 0, 69],
        0.025: [0, 47, 0, 0, 0, 69],
        0.050: [0, 46, 0, 0, 0, 69],
        0.100: [0, 47, 0, 1, 1, 78],
    },
    index=quantile_alpha_results.index,
)
gaussian_quantile_alpha_results.columns.name = r"quantile level $\alpha$"
display(gaussian_quantile_alpha_results)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True, constrained_layout=True)
for ax, table, title, bic_count in [
    (axes[0], quantile_alpha_results, "Uniform innovations", 100),
    (axes[1], gaussian_quantile_alpha_results, "Gaussian innovations", 99),
]:
    ax.plot(table.columns, table.loc["Relative slope drop (material)"],
            "o-", lw=2, label="material relative slope")
    ax.plot(table.columns, table.loc["Plateau onset (10%)"],
            "o-", lw=2, label="plateau onset")
    ax.axhline(bic_count, color="black", ls="--", lw=1.5, label=f"BIC: {bic_count}/100")
    ax.set(title=title, xlabel=r"quantile trimming level $\alpha$", ylim=(0, 103))
    ax.legend(fontsize=8)
axes[0].set_ylabel("correct selections out of 100")
plt.show()

## Heavy-tailed innovations: Student-$t_2$ and Cauchy

The AR(3) experiment was repeated with Student-$t_2$ innovations, which have infinite variance. To make central scales comparable, the innovations were scaled to have the same median absolute deviation as $N(0,0.25^2)$. Quantile trimming dramatically rescues the geometric plateau rule: recovery rises from 7/100 for the raw range to 94/100 at $\alpha=0.10$. Nevertheless, BIC still selected the correct order in 97/100 paths.

A sharper Cauchy stress test (no finite mean or variance) gave BIC 98/100. At $\alpha=0.10$ the relative-slope and plateau rules achieved 49/100 and 55/100; at $\alpha=0.20$ both achieved 60/100. Thus heavy tails create a real role for quantile width, but these experiments do not show it outperforming BIC.

In [ ]:
student_t2_alpha_results = pd.DataFrame(
    {
        0.000: [8, 27, 7, 28, 20, 7],
        0.010: [0, 53, 0, 0, 0, 70],
        0.025: [0, 66, 0, 1, 1, 85],
        0.050: [0, 70, 0, 0, 1, 90],
        0.100: [0, 74, 0, 0, 0, 94],
        0.150: [0, 82, 0, 1, 0, 92],
        0.200: [0, 85, 0, 0, 2, 93],
    },
    index=quantile_alpha_results.index,
)
student_t2_alpha_results.columns.name = r"quantile level $\alpha$"
display(student_t2_alpha_results)

cauchy_summary = pd.DataFrame(
    {"relative slope drop": [49, 60], "plateau onset": [55, 60]},
    index=pd.Index([0.10, 0.20], name=r"quantile level $\alpha$"),
)
display(cauchy_summary)

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(student_t2_alpha_results.columns,
        student_t2_alpha_results.loc["Relative slope drop (material)"],
        "o-", lw=2, label="material relative slope")
ax.plot(student_t2_alpha_results.columns,
        student_t2_alpha_results.loc["Plateau onset (10%)"],
        "o-", lw=2, label="plateau onset")
ax.axhline(97, color="black", ls="--", lw=1.5, label="BIC: 97/100")
ax.set(xlabel=r"quantile trimming level $\alpha$",
       ylabel="correct selections out of 100", ylim=(0, 103),
       title=r"AR(3) order recovery with Student-$t_2$ innovations")
ax.legend()
plt.show()